# 🧩 Mini-Lab: Narrow Exceptions

**Module: Trust but Verify: Validation, Config, Errors, Logs** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Identify** the dangers of bare `except` clauses that silently swallow errors
2. **Define** narrow custom exception types that communicate what went wrong
3. **Replace** broad exception handling with specific catches that fail loudly

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Narrow Exception Handling | Custom exception types beat bare `except`, never swallow errors. Catch only what you can handle, add context, then raise or crash so failures are visible and fixable. |
| Silent Failure | An empty filter result or coerced dtype can silently change what your code is operating on, producing confident, wrong output without any exception or traceback. |


## The Problem: Silent Failures

A **bare `except`** catches *every* exception — including bugs you need to see. The program keeps running, but with wrong or missing data. You only discover the problem hours later (or never).

| Pattern | Risk |
|---------|------|
| `except:` | Catches `KeyboardInterrupt`, `SystemExit`, *everything* |
| `except Exception:` | Catches all bugs — `TypeError`, `NameError`, `IndexError` — and hides them |
| `except Exception: pass` | The worst: swallows the error *and* produces no output |

## Step 1 — See a Silent Failure in Action

Below is a function that parses a price from a string. It has a **bug** (a typo in the variable name), but the bare `except` hides it completely.

In [1]:
def parse_price_bad(text: str) -> float:
    """Parse a dollar price from text — BAD version with bare except."""
    try:
        cleaned = text.replace("$", "").replace(",", "").strip()
        price = float(cleaned)
        return pirce  # ← typo! should be 'price'
    except:
        return 0.0  # silently returns 0 for ALL errors


# This LOOKS like it works — but every result is wrong
prices = ["$19.99", "$1,250.00", "free", "$42.50"]
results = [parse_price_bad(p) for p in prices]
print("Parsed prices:", results)
print("Total:        ", sum(results))

Parsed prices: [0.0, 0.0, 0.0, 0.0]
Total:         0.0


Every price came back as `0.0` — even the valid ones. The `NameError` from the typo (`pirce`) was silently swallowed. In a real pipeline, you'd compute a total of `$0.00` and never know why.

## Step 2 — Define a Narrow Custom Exception

Instead of catching everything, we define a custom exception that describes the *specific* failure we can anticipate: a price string that can't be converted to a number.

In [2]:
class PriceParsingError(ValueError):
    """Raised when a price string cannot be parsed into a number."""

    def __init__(self, raw_value: str, reason: str):
        self.raw_value = raw_value
        self.reason = reason
        super().__init__(f"Cannot parse price '{raw_value}': {reason}")


print("Custom exception defined:", PriceParsingError.__mro__)

Custom exception defined: (<class '__main__.PriceParsingError'>, <class 'ValueError'>, <class 'Exception'>, <class 'BaseException'>, <class 'object'>)


Notice two design choices:

1. **Inherits from `ValueError`**, not `Exception` — this tells readers *what kind* of problem it is
2. **Carries context** (`raw_value`, `reason`) — the error message tells you exactly what input failed and why

## Step 3 — Catch Narrow, Let Bugs Propagate

Now we rewrite `parse_price` to catch **only** `ValueError` (the one error we expect from `float()`) and raise our custom exception. Any other error — like the typo — will propagate and be visible immediately.

In [3]:
def parse_price(text: str) -> float:
    """Parse a dollar price from text — GOOD version with narrow except."""
    cleaned = text.replace("$", "").replace(",", "").strip()
    if not cleaned:
        raise PriceParsingError(text, "empty after cleaning")
    try:
        return float(cleaned)
    except ValueError:
        raise PriceParsingError(text, f"'{cleaned}' is not a valid number")

Key differences from the bad version:

- The `try` block wraps **only** the `float()` call — the smallest possible scope
- We catch **only** `ValueError` — the one exception `float()` raises on bad input
- A typo, `TypeError`, or any other bug is **not caught** — it crashes loudly so you fix it

## Step 4 — Test with Good and Bad Inputs

Let's run the same price list through the improved function. This time, valid prices succeed and invalid ones produce a clear, specific error.

In [4]:
prices = ["$19.99", "$1,250.00", "free", "$42.50", ""]

for p in prices:
    try:
        result = parse_price(p)
        print(f"  ✅ '{p}' → {result}")
    except PriceParsingError as exc:
        print(f"  ❌ {exc}")

  ✅ '$19.99' → 19.99
  ✅ '$1,250.00' → 1250.0
  ❌ Cannot parse price 'free': 'free' is not a valid number
  ✅ '$42.50' → 42.5
  ❌ Cannot parse price '': empty after cleaning


Valid prices are parsed correctly; invalid ones produce a `PriceParsingError` with the exact input and reason. No silent zeros.

## Step 5 — The Caller Decides What to Do

With narrow exceptions, the **caller** can choose the right recovery strategy: skip, use a default, or abort. The function doesn't make that decision for you.

In [5]:
prices = ["$19.99", "free", "$42.50", "N/A", "$8.00"]

# Strategy: skip bad values, collect only valid ones
valid_prices = []
skipped = []

for p in prices:
    try:
        valid_prices.append(parse_price(p))
    except PriceParsingError:
        skipped.append(p)

print(f"Valid prices: {valid_prices}")
print(f"Skipped:      {skipped}")
print(f"Total:        ${sum(valid_prices):.2f}")

Valid prices: [19.99, 42.5, 8.0]
Skipped:      ['free', 'N/A']
Total:        $70.49


## Step 6 — Side-by-Side Comparison

Let's summarize the two approaches.

In [6]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

md("""
| | ❌ Bare `except` | ✅ Narrow `except` |
|---|---|---|
| **What it catches** | Everything — bugs, typos, system signals | Only the specific error you expect |
| **Bugs** | Hidden — program continues with wrong data | Visible — crash immediately so you fix them |
| **Error message** | None (or generic) | Specific: what input failed and why |
| **Caller control** | None — function decides to return a default | Full — caller picks the recovery strategy |
| **Debugging** | "Why is the total $0?" (hours later) | `PriceParsingError: 'free' is not a valid number` (immediately) |
""")


| | ❌ Bare `except` | ✅ Narrow `except` |
|---|---|---|
| **What it catches** | Everything — bugs, typos, system signals | Only the specific error you expect |
| **Bugs** | Hidden — program continues with wrong data | Visible — crash immediately so you fix them |
| **Error message** | None (or generic) | Specific: what input failed and why |
| **Caller control** | None — function decides to return a default | Full — caller picks the recovery strategy |
| **Debugging** | "Why is the total $0?" (hours later) | `PriceParsingError: 'free' is not a valid number` (immediately) |


## 🎯 Summary

### Key Takeaways

1. **Silent failure** — a bare `except` swallows every error including bugs, letting the program continue with wrong data and no warning
2. **Narrow exception handling** — catch only the specific exception type you can recover from (e.g., `ValueError` from `float()`), and let everything else propagate
3. **Custom exception types** — defining a class like `PriceParsingError(ValueError)` carries context (what input, why it failed) and makes errors self-documenting
4. **Caller decides recovery** — the function raises; the caller chooses whether to skip, default, retry, or abort